In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_customers = spark.table("digital_banking.silver.silver_customers")

df_dim_customer = df_silver_customers.select(
    # Primary Key
    F.col("customer_id").alias("customer_key"),
    
    # Customer Identification
    F.col("customer_id"),
    
    # Personal Information
    F.col("first_name"),
    F.col("last_name"),
    F.concat_ws(" ", F.col("first_name"), F.col("last_name")).alias("full_name"),
    F.col("date_of_birth"),
    
    # Derived Age Metrics
    F.floor(F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25).alias("age"),
    F.when(F.floor(F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25) < 25, "18-24")
     .when(F.floor(F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25) < 35, "25-34")
     .when(F.floor(F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25) < 45, "35-44")
     .when(F.floor(F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25) < 55, "45-54")
     .when(F.floor(F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25) < 65, "55-64")
     .otherwise("65+").alias("age_group"),
    
    # Contact Information
    F.col("email"),
    F.col("phone"),
    
    # Address Information
    F.col("address"),
    F.col("city"),
    F.col("state"),
    F.col("postal_code"),
    F.concat_ws(", ", F.col("city"), F.col("state"), F.col("postal_code")).alias("full_address"),
    
    # Customer Attributes
    F.coalesce(F.col("customer_segment"), F.lit("Unknown")).alias("customer_segment"),
    F.coalesce(F.col("customer_status"), F.lit("Unknown")).alias("customer_status"),
    
    # Temporal Attributes
    F.col("registration_date"),
    F.floor(F.datediff(F.current_date(), F.col("registration_date")) / 365.25).alias("customer_tenure_years"),
    
    # Audit Columns
    F.col("updated_at").alias("source_updated_at"),
    F.current_timestamp().alias("dimension_created_at"),
    F.current_timestamp().alias("dimension_updated_at")
)

# Write to gold layer as a managed Delta table
df_dim_customer.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("digital_banking.gold.dim_customer")

print(f"Total customers: {df_dim_customer.count()}")
# Display 
display(df_dim_customer.limit(10))